# JalDrishti — Phase 2: U-Net ResNet34 Flood Segmentation Training

This notebook trains the JalDrishti Phase 2 U-Net model on the official **Sen1Floods11** hand-labelled dataset using Sentinel-1 SAR imagery.

### Platform Setup:
- **Google Colab**: Select **Runtime** → **Change runtime type** → **T4 GPU**.
- **Kaggle**: In the notebook settings sidebar, select **Accelerator** → **GPU T4 x2** or **P100**.

### Rules & Integrity:
- Trains strictly on the **train** split (252 chips).
- Tunes hyperparameters strictly on the **valid** split (89 chips).
- Evaluates strictly on the held-out **test** split (90 chips) and **Bolivia** split (15 chips).
- Ground truth `-1` (cloud/no-data) pixels are masked in both loss and metrics.

In [ ]:
# 1. Install dependencies
!pip install -q segmentation-models-pytorch tifffile requests scipy scikit-image

In [ ]:
# 2. Check GPU environment
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

In [ ]:
# 3. Dataset loader with train statistics normalization and no-data masking
import os
import random
from pathlib import Path
import numpy as np
import requests
import tifffile
from torch.utils.data import Dataset, DataLoader

SEN1FLOODS11_GCS_BASE = "https://storage.googleapis.com/sen1floods11/v1.1"

# Statistics computed strictly from the train split
TRAIN_STATS = {
    "vv_mean": -9.6400,
    "vv_std": 2.7252,
    "vh_mean": -15.9661,
    "vh_std": 3.1531,
    "diff_mean": 6.3261,
    "diff_std": 2.8825,
}

def load_split_chips(split_name: str):
    fname_map = {
        "train": "flood_train_data.csv",
        "valid": "flood_valid_data.csv",
        "test": "flood_test_data.csv",
        "bolivia": "flood_bolivia_data.csv",
    }
    url = f"{SEN1FLOODS11_GCS_BASE}/splits/flood_handlabeled/{fname_map[split_name]}"
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    pairs = []
    for line in resp.text.strip().split("\n"):
        parts = [p.strip() for p in line.split(",") if p.strip()]
        if len(parts) >= 2:
            pairs.append((parts[0], parts[1]))
    return pairs

class Sen1Floods11Dataset(Dataset):
    def __init__(self, split="train", cache_dir="data/cache/sen1floods11", crop_size=256, is_train=True, max_samples=None):
        self.split = split
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.crop_size = crop_size
        self.is_train = is_train
        self.pairs = load_split_chips(split)
        if max_samples:
            self.pairs = self.pairs[:max_samples]

    def __len__(self):
        return len(self.pairs)

    def _fetch_file(self, subpath: str, filename: str) -> bytes:
        local_path = self.cache_dir / filename
        if local_path.is_file():
            return local_path.read_bytes()
        url = f"{SEN1FLOODS11_GCS_BASE}/data/flood_events/HandLabeled/{subpath}/{filename}"
        r = requests.get(url, timeout=25)
        r.raise_for_status()
        local_path.write_bytes(r.content)
        return r.content

    def __getitem__(self, idx):
        s1_name, lbl_name = self.pairs[idx]
        import io
        s1_bytes = self._fetch_file("S1Hand", s1_name)
        lbl_bytes = self._fetch_file("LabelHand", lbl_name)
        s1 = tifffile.imread(io.BytesIO(s1_bytes)) # [2, 512, 512]
        lbl = tifffile.imread(io.BytesIO(lbl_bytes)) # [512, 512]

        vv = s1[0].astype(np.float32)
        vh = s1[1].astype(np.float32)

        vv_clean = np.where(np.isfinite(vv), vv, -10.0)
        vh_clean = np.where(np.isfinite(vh), vh, -16.0)
        diff = vv_clean - vh_clean

        vv_norm = (vv_clean - TRAIN_STATS["vv_mean"]) / TRAIN_STATS["vv_std"]
        vh_norm = (vh_clean - TRAIN_STATS["vh_mean"]) / TRAIN_STATS["vh_std"]
        diff_norm = (diff - TRAIN_STATS["diff_mean"]) / TRAIN_STATS["diff_std"]

        img = np.stack([vv_norm, vh_norm, diff_norm], axis=0) # [3, 512, 512]
        mask = (lbl == 1).astype(np.float32)[np.newaxis, :, :]
        valid = (lbl >= 0).astype(np.float32)[np.newaxis, :, :]

        if self.is_train and self.crop_size < 512:
            H, W = 512, 512
            top = random.randint(0, H - self.crop_size)
            left = random.randint(0, W - self.crop_size)
            img = img[:, top:top+self.crop_size, left:left+self.crop_size]
            mask = mask[:, top:top+self.crop_size, left:left+self.crop_size]
            valid = valid[:, top:top+self.crop_size, left:left+self.crop_size]

            if random.random() > 0.5:
                img = np.flip(img, axis=2).copy()
                mask = np.flip(mask, axis=2).copy()
                valid = np.flip(valid, axis=2).copy()
            if random.random() > 0.5:
                img = np.flip(img, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
                valid = np.flip(valid, axis=1).copy()
            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k, axes=(1, 2)).copy()
                mask = np.rot90(mask, k, axes=(1, 2)).copy()
                valid = np.rot90(valid, k, axes=(1, 2)).copy()
            if random.random() > 0.5:
                noise = np.random.normal(0.0, 0.05, img.shape).astype(np.float32)
                img = img + noise

        return torch.from_numpy(img), torch.from_numpy(mask), torch.from_numpy(valid)

print("Dataset class ready.")

In [ ]:
# 4. Loss Function: Masked Dice + BCE (masks -1 no-data pixels)
import torch.nn as nn
import torch.nn.functional as F

class MaskedDiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.smooth = smooth

    def forward(self, logits, targets, valid_masks):
        probs = torch.sigmoid(logits)
        v = valid_masks.bool()
        if v.sum() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        p_valid = probs[v]
        t_valid = targets[v]

        bce = F.binary_cross_entropy(p_valid, t_valid)
        intersection = (p_valid * t_valid).sum()
        dice = 1.0 - (2.0 * intersection + self.smooth) / (p_valid.sum() + t_valid.sum() + self.smooth)
        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice

print("MaskedDiceBCELoss defined.")

In [ ]:
# 5. Instantiate U-Net ResNet34
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

print(f"U-Net ResNet34 loaded with {sum(p.numel() for p in model.parameters()):,} parameters.")

In [ ]:
# 6. Training Loop (Cosine Annealing LR & Validation IoU Model Checkpointing)
train_ds = Sen1Floods11Dataset(split="train", is_train=True, crop_size=256)
val_ds = Sen1Floods11Dataset(split="valid", is_train=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

criterion = MaskedDiceBCELoss(bce_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

best_val_iou = 0.0
best_weights_path = "unet_resnet34.pth"

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for imgs, tgts, valids in train_loader:
        imgs, tgts, valids = imgs.to(device), tgts.to(device), valids.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), tgts, valids)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # Validation evaluation
    model.eval()
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for imgs, tgts, valids in val_loader:
            imgs, tgts, valids = imgs.to(device), tgts.to(device), valids.to(device)
            preds = (torch.sigmoid(model(imgs)) > 0.5)
            v = valids.bool()
            y_t = tgts[v].bool()
            y_p = preds[v].bool()
            tp += (y_p & y_t).sum().item()
            fp += (y_p & ~y_t).sum().item()
            fn += (~y_p & y_t).sum().item()

    val_iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    val_f1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
    print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss/len(train_loader):.4f} | Val IoU: {val_iou:.4f} | Val F1: {val_f1:.4f}")

    if val_iou >= best_val_iou:
        best_val_iou = val_iou
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_iou": val_iou,
            "architecture": "unet_resnet34",
        }, best_weights_path)
        print(f"  --> Saved best weights to {best_weights_path} (Val IoU: {val_iou:.4f})")

print("Training complete! Best Val IoU:", best_val_iou)

In [ ]:
# 7. Download weights to bring back to JalDrishti
try:
    from google.colab import files
    files.download(best_weights_path)
    print("Downloading weights to local machine...")
except ImportError:
    print(f"Model weights saved locally to {best_weights_path}. If on Kaggle, download from the Output sidebar.")

### How to Bring the Weights Back to JalDrishti:
1. Save the downloaded `unet_resnet34.pth` into your repository at `pipeline/weights/unet_resnet34.pth`.
2. Run inference: `python pipeline/infer_unet.py`.
3. Run evaluation: `python pipeline/evaluate.py`.
4. `pipeline/weights/` is ignored by Git in `.gitignore` to avoid bloating the repository.